# FASE 6 - Multimodal: Imagem

Status: proposta inicial  
Data: 2026-03-10  
Escopo deste documento: planejamento apenas. Nenhuma implementacao foi executada a partir desta proposta.



## Objetivo da fase

Entregar uma base local de geracao de imagens com Stable Diffusion capaz de:

- gerar imagens a partir de contexto em texto;
- gerar imagens a partir de uma ou mais imagens de referencia;
- manter consistencia visual entre imagens do mesmo projeto;
- produzir conjuntos visuais reutilizaveis para:
  - ilustracoes de historias;
  - arte para musica;
  - colecoes visuais coerentes para divulgacao ou narrativa.



## Resultado esperado

Ao final da fase 6, o projeto deve conseguir:

- rodar geracao local sem dependencia de API paga por imagem;
- aceitar prompt textual + referencias visuais + regras de estilo;
- produzir lotes de imagens com metadados reprodutiveis;
- manter um "visual bible" por projeto para preservar estilo, paleta, enquadramento e identidade;
- salvar artefatos em estrutura compativel com o padrao atual de `artifacts/`.

Observacao importante: "sem limites de quantidade" deve ser entendido como sem limite artificial de provedor. Na pratica, o limite passa a ser hardware, tempo de processamento e espaco em disco.



## Premissas tecnicas

- O repositorio atual ja usa Python, Poetry, Torch, Pillow e uma estrutura modular em `tools/`.
- A fase 6 deve se integrar ao sistema de artefatos ja existente, em vez de criar um fluxo paralelo isolado.
- O ambiente alvo e local, provavelmente Windows, com suporte ideal a GPU NVIDIA.
- CPU-only pode servir para smoke test funcional, mas nao e uma meta realista para uso diario com boa resolucao.
- O primeiro perfil operacional a suportar bem esta fase sera o de GPUs na classe da `RTX 2060 6 GB`, com `16 GB` de RAM do sistema.
- Isso muda o desenho do MVP: o fluxo padrao deve priorizar estabilidade e throughput, usando resolucao menor, lote pequeno e upscale posterior.



## Recomendacao de arquitetura

### Direcao principal

Usar `diffusers` como backend programatico principal e adotar `ComfyUI` como interface local principal para prototipagem e operacao. Para o perfil inicial de hardware do projeto, o fluxo padrao deve usar checkpoints `SD 1.5-class`, reservando `SDXL` para refinamento ou imagem final selecionada.

### Motivo da recomendacao

- `diffusers` encaixa melhor no stack Python atual do projeto.
- A automacao de lotes, manifests, seeds e exportacao de artefatos fica mais previsivel dentro do codigo do ScoreIA.
- `ComfyUI` facilita tuning fino de workflows com pouca VRAM e torna mais facil isolar gargalos de `img2img`, `inpainting`, `IP-Adapter` e `ControlNet`.
- Em `6 GB` de VRAM, `SD 1.5-class` com `img2img/inpainting`, `IP-Adapter` e no maximo `1 ControlNet` por vez e muito mais realista como fluxo padrao do que `SDXL` ou `FLUX` como base diaria.
- A integracao fica hibrida e pragmatica:
  - core de producao e reproducibilidade no repositorio;
  - operacao visual e ajuste de workflow no `ComfyUI`.

### Regra operacional do MVP

- gerar rascunhos em `512x512`;
- usar `batch size = 1`;
- fazer variacoes em sequencia, nao em paralelo;
- usar `img2img` ou `inpainting` apenas na imagem escolhida;
- aplicar upscale apenas no final;
- evitar empilhar `SDXL + ControlNet + IP-Adapter + inpainting` no mesmo workflow.



## Blocos funcionais da fase

### 1. Prompt and context engine

Responsavel por transformar:

- texto livre;
- contexto narrativo;
- briefing de musica/produto;
- referencias anteriores do projeto;

em uma especificacao visual consistente contendo:

- prompt base;
- negative prompt;
- estilo visual;
- paleta;
- camera/enquadramento;
- lista de personagens ou elementos recorrentes;
- variacoes permitidas;
- restricoes de consistencia.

### 2. Local image runtime

Camada que carrega checkpoints, LoRAs, IP-Adapters e ControlNets, executa a inferencia e grava os artefatos.

### 3. Reference conditioning

Camada para:

- img2img;
- uso de imagem de referencia por estilo;
- uso de imagem de referencia por composicao;
- uso de imagem de referencia por personagem/face, quando aplicavel.

### 4. Consistency layer

Camada que tenta manter continuidade visual entre imagens do mesmo projeto usando:

- visual bible;
- seeds relacionadas;
- presets de prompt;
- mesmas configuracoes de modelo/sampler;
- modulos de referencia;
- LoRA opcional por projeto, apenas se a consistencia basica nao bastar.

### 5. Batch and pack generation

Camada para gerar lotes como:

- storyboard de historia;
- kit visual para musica;
- conjunto de pecas com mesma identidade visual.

### 6. Artifact and UI integration

Camada de persistencia e interface:

- salvar imagens, manifestos e referencias;
- gerar contact sheet ou preview grid;
- disponibilizar fluxo na webapp ja existente.



## Pacotes Python propostos

### Ja presentes e reaproveitaveis

- `torch`
- `torchvision`
- `numpy`
- `pillow`

### Nucleo recomendado

- `diffusers`
  - backend principal para Stable Diffusion local, img2img, pipelines SDXL e componentes de upscaling;
- `transformers`
  - tokenizers e componentes de modelos usados pelo ecossistema do `diffusers`;
- `accelerate`
  - gerenciamento de device, precision e carga de modelo;
- `safetensors`
  - carregamento seguro e padrao para checkpoints e LoRAs;
- `huggingface_hub`
  - download e versionamento local de modelos, quando aplicavel;
- `compel`
  - composicao mais controlada de prompts e pesos;
- `opencv-python`
  - preprocessamento de imagem e apoio a pipelines de referencia;
- `pydantic`
  - validacao de payloads de geracao, presets e manifests.

### Recomendados para referencia e consistencia

- `controlnet_aux`
  - preprocessadores como canny, depth, openpose e lineart;
- `scikit-image`
  - metricas simples e utilitarios de comparacao/transformacao de imagem;
- `imageio`
  - exportacao de grids, previews e eventuais sequencias.

### Opcionais por necessidade real

- `xformers`
  - melhora de memoria/performance em GPU; util, mas pode trazer friccao de instalacao no Windows;
- `peft`
  - suporte a LoRA e adaptacoes leves, se o projeto evoluir para ajuste fino;
- `insightface`
  - embeddings faciais para consistencia de identidade em personagens humanos;
- `onnxruntime-gpu` ou `onnxruntime`
  - apoio a face analysis / identity locking quando `insightface` entrar;
- `realesrgan` ou upscaler equivalente
  - apenas se a etapa de upscaling local virar requisito obrigatorio logo no inicio.

### O que eu nao colocaria no primeiro corte

- treinamento completo de modelo customizado;
- fine-tuning pesado como dependencia inicial;
- empilhar muitas bibliotecas de identidade facial antes do fluxo base funcionar.



## Ferramentas externas que podem ser necessarias

### Obrigatorias ou quase obrigatorias

- Driver NVIDIA atualizado e stack CUDA compativel com a versao de `torch` em uso.
- Espaco em disco dedicado para checkpoints, ControlNets, IP-Adapters, LoRAs e outputs.

### Fortemente recomendadas

- `ComfyUI`
  - interface local principal para operacao e depuracao de workflows no perfil de `6 GB` de VRAM;
- `Hugging Face CLI`
  - download controlado de modelos e autenticacao quando algum peso exigir aceite de licenca;
- `MobileSAM`
  - geracao de mascaras de edicao em um stack mais leve do que alternativas maiores de segmentacao.

### Opcionais

- `Git LFS`
  - se o projeto decidir versionar presets pesados ou pequenos modelos auxiliares fora do fluxo comum de Git;
- ferramenta local de upscaling dedicada
  - para subir resolucao apenas da imagem aprovada, sem gastar VRAM no fluxo inteiro;
- viewer externo para comparacao lado a lado de lotes gerados.



## Modelos e pesos recomendados

### Base operacional para 6 GB de VRAM

- 1 checkpoint principal `SD 1.5-class` para uso diario.
- 1 modelo de `inpainting` compativel com a mesma familia.
- 1 `IP-Adapter` compativel com a familia principal.
- 1 conjunto minimo de `ControlNet`, usando um modulo por vez:
  - canny;
  - depth;
  - lineart ou openpose, conforme o caso.
- `MobileSAM` para mascaras de edicao.

### Base de qualidade pontual

- 1 checkpoint `SDXL` reservado para imagem final selecionada ou casos em que o ganho visual justifique o custo de VRAM e tempo.
- upscaler x2/x4 para entrega final.

### Opcionais depois do MVP

- `InstructPix2Pix` para edicoes guiadas por instrucao textual em imagens ja escolhidas;
- 1 checkpoint mais estilizado para arte musical ou editorial;
- LoRAs especificas por projeto.

### O que nao entra como fluxo principal nesta maquina

- `FLUX Fill/Kontext` como padrao local nesta primeira iteracao;
- workflows que combinem `SDXL + ControlNet + IP-Adapter + inpainting` na mesma execucao.



## Estrutura de pastas recomendada

### Pesos de modelo

Os pesos nao devem entrar no Git normal do repositorio. O ideal e usar um diretorio configuravel, por exemplo:

```text
models/
  checkpoints/
  controlnet/
  ip_adapter/
  loras/
  upscalers/
  vae/
```

Idealmente isso deve ser controlado por variavel de ambiente, por exemplo `SCOREIA_MODELS_DIR`.

### Artefatos de execucao

Expandir a estrutura atual de `artifacts/` para incluir imagens:

```text
artifacts/
  run_<timestamp>/
    images/
      generated/
      references/
      selected/
      contact_sheets/
      metadata/
```

Arquivos esperados por execucao:

- `image_manifest_<timestamp>.json`
- `prompt_book_<timestamp>.json`
- `visual_bible_<timestamp>.json`
- `contact_sheet_<timestamp>.png`



## Arquivos e scripts propostos

### Modulos em `tools/`

- `tools/image_config.py`
  - schemas, defaults, presets de resolucao e leitura de env vars;
- `tools/image_models.py`
  - registry de checkpoints, LoRAs, ControlNets e caminhos locais;
- `tools/image_prompt_builder.py`
  - transforma contexto em prompt estruturado e negative prompt;
- `tools/image_reference.py`
  - normaliza imagens de referencia e preprocessa entradas;
- `tools/image_generation.py`
  - text2img, img2img, reference-based generation;
- `tools/image_consistency.py`
  - visual bible, seeds de familia, score de similaridade e validacoes;
- `tools/image_batches.py`
  - geracao de story packs, music packs e visual sets;
- `tools/image_manifest.py`
  - salva metadados, hashes, prompts, parametros e relacao entre outputs;
- `tools/image_upscale.py`
  - opcional, se upscaling entrar no MVP;
- `tools/image_smoke.py`
  - smoke test minimo do stack multimodal.

### Ajustes em modulos existentes

- `tools/artifacts.py`
  - incluir caminhos padrao para outputs de imagem;
- `tools/artifact_export.py`
  - registrar manifests e hashes de imagens;
- `tools/webapp/app.py`
  - adicionar entradas para geracao de imagem e consulta de artefatos;
- `main.py`
  - expor comandos iniciais de imagem, se a CLI continuar sendo porta de entrada.

### Scripts de entrada

- `scripts/generate_images.py`
  - CLI generica para `text2img`, `img2img` e `reference`;
- `scripts/generate_story_pack.py`
  - recebe historia/roteiro e gera lista de cenas + imagens;
- `scripts/generate_music_pack.py`
  - gera capa, variacoes e pecas promocionais a partir de briefing musical;
- `scripts/validate_phase6_stack.py`
  - verifica modelos, ambiente, paths e smoke de inferencia.

### Testes recomendados

- `tests/test_image_prompt_builder.py`
  - garante que o prompt estruturado preserve campos criticos;
- `tests/test_image_manifest.py`
  - garante integridade de manifests;
- `tests/test_image_consistency.py`
  - valida regra de visual bible e persistencia de seeds/presets;
- `tests/test_image_config.py`
  - valida resolucao de paths e configuracoes.



## Fluxos de uso previstos

### 1. Ilustracoes de historias

Entrada:

- sinopse;
- personagens;
- tom visual;
- referencias opcionais;
- numero de cenas.

Saida:

- shot list;
- visual bible;
- lote de imagens por cena;
- contact sheet para revisao.

### 2. Arte para musica

Entrada:

- genero;
- clima;
- letra ou resumo da musica;
- referencias de capa/estilo;
- formatos desejados.

Saida:

- capa principal;
- variacoes;
- pecas quadradas e verticais;
- prompts e seeds preservados.

### 3. Conjuntos visuais

Entrada:

- identidade visual do projeto;
- referencia de estilo;
- conjunto de temas ou pecas.

Saida:

- varias imagens consistentes no mesmo universo estetico;
- manifest com relacao entre variantes;
- selecao final pronta para exportacao.



## Estrategia de consistencia visual

Consistencia visual nao deve depender de um unico truque. A proposta e combinar camadas:

1. `visual bible` por projeto
   - estilo, paleta, materiais, iluminacao, enquadramentos, negative prompt, checkpoint e preset;
2. `prompt templates`
   - termos fixos e termos variaveis bem separados;
3. `seed families`
   - sementes base por personagem, cena ou campanha;
4. `reference conditioning`
   - IP-Adapter e ControlNet quando houver referencia concreta;
5. `LoRA opcional`
   - apenas se a consistencia de personagens/estilo ficar insuficiente no fluxo base.



## Ordem sugerida de implementacao

### Etapa 6.1 - Infraestrutura base

- config de modelos;
- paths de artefatos;
- manifest de imagem;
- smoke test de carga de pipeline.

### Etapa 6.2 - Geracao por texto

- comecar por `SD 1.5-class`;
- presets de resolucao com foco em `512x512` e `768x768` apenas para fluxos estaveis;
- persistencia de prompt, seed e modelo.

### Etapa 6.3 - Geracao por referencia

- `img2img`;
- `inpainting`;
- `IP-Adapter`;
- preprocessadores de `ControlNet` basicos;
- limitar o workflow base a `1` modulo pesado por vez.

### Etapa 6.4 - Consistencia visual

- visual bible;
- seed families;
- validadores simples de coerencia;
- reuso da imagem aprovada como ancora de estilo.

### Etapa 6.5 - Pacotes de uso final

- gerar lotes pequenos em sequencia;
- selecionar as melhores imagens antes do refinamento;
- aplicar upscale apenas nas imagens aprovadas;
- story pack;
- music pack;
- visual set pack.

### Etapa 6.6 - Webapp e exportacao

- UI minima;
- preview grid;
- manifests integrados ao fluxo atual.



## Requisitos de hardware recomendados

### Perfil operacional atual do projeto

- CPU `Intel i7-10750H`;
- `16 GB` de RAM;
- GPU `RTX 2060 6 GB`;
- esse perfil suporta bem um MVP com `SD 1.5-class`, `img2img/inpainting`, `IP-Adapter`, `1 ControlNet por vez` e upscale posterior.

### Parametros operacionais recomendados para esse perfil

- rascunhos em `512x512`;
- subir para `768x768` apenas quando o workflow estiver estavel;
- `batch size = 1`;
- gerar variacoes e lotes pequenos em sequencia;
- fechar apps pesados durante inferencia;
- deixar `SDXL` para uso pontual, nao como padrao diario.

### Minimo pragmatico para o MVP nesta maquina

- GPU NVIDIA com `6 GB` de VRAM;
- `16 GB` de RAM;
- pelo menos `50 GB` livres em SSD para modelos e artefatos.

### Ideal para uso confortavel

- GPU com `12 GB` ou mais de VRAM;
- `32 GB` de RAM;
- SSD dedicado para `models/` e `artifacts/`.



## Riscos principais

- incompatibilidade entre CUDA, Torch e pacotes auxiliares no Windows;
- saturacao de VRAM ao combinar `SDXL`, `ControlNet`, `IP-Adapter` e `inpainting` no mesmo fluxo;
- lentidao ou travamento ao tentar lotes grandes ou resolucoes altas demais na `RTX 2060 6 GB`;
- perda de produtividade se o projeto tentar usar `FLUX` como fluxo local principal nesta maquina;
- pressao na RAM do sistema (`16 GB`) quando houver navegador, IDE e inferencia local pesados ao mesmo tempo.



## Criterios de aceite da fase

- gerar imagens locais em pelo menos tres modos:
  - texto para imagem;
  - imagem de referencia + texto;
  - lote consistente por projeto;
- salvar seeds, prompts, modelo e parametros em manifest;
- suportar pelo menos um fluxo de historia e um fluxo de musica;
- manter organizacao de artefatos compativel com o padrao atual do projeto;
- permitir reproducao basica de uma imagem a partir do manifest salvo;
- validar um fluxo estavel no perfil `RTX 2060 6 GB`, com rascunho em baixa resolucao e upscale apenas na imagem aprovada.



## Decisoes recomendadas para aprovar antes da implementacao

- adotar `diffusers` como engine principal;
- tratar `ComfyUI` como interface local principal do MVP;
- manter pesos fora do Git do repositorio;
- comecar com `SD 1.5-class + img2img/inpainting + IP-Adapter + 1 ControlNet por vez + MobileSAM`;
- formalizar o workflow `rascunho pequeno -> selecao -> refinamento -> upscale`;
- deixar `SDXL` para imagem final selecionada, nao como fluxo diario;
- deixar `FLUX`, LoRA pesada e fine-tuning como expansao posterior, nao como prerequisito do MVP.

